In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import qnt.data as qndata

# ============================================================
# 1) Load Quantiacs data + pick random liquid assets
# ============================================================
def load_quantiacs_universe(seed=7, n_assets=50):
    data = qndata.stocks.load_data()  # DataArray(field, time, asset)
    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    liquid_last = is_liquid.isel(time=-1).fillna(0).astype(bool)
    liquid_assets = close.asset.values[liquid_last.values]

    if len(liquid_assets) < n_assets:
        raise ValueError(f"Not enough liquid assets ({len(liquid_assets)}) to pick {n_assets}.")

    rng = np.random.default_rng(seed)
    chosen_assets = rng.choice(liquid_assets, size=n_assets, replace=False)
    return data, chosen_assets


# ============================================================
# 2) Returns matrix R (daily) from close
# ============================================================
def compute_returns_matrix(data, chosen_assets, use_log_returns=False):
    close = data.sel(field="close").sel(asset=chosen_assets)

    if use_log_returns:
        rets = np.log(close / close.shift(time=1))
    else:
        rets = close / close.shift(time=1) - 1

    rets = rets.dropna("time")  # DataArray(time, asset)
    times = rets.time.values
    R = rets.values  # (T, n)
    return times, R


# ============================================================
# 3) Rolling Sigma_t (covariance) - annualized
# ============================================================
def compute_sigma_series(R, lookback=252, annualize=252):
    T, n = R.shape
    if T <= lookback + 1:
        raise ValueError(f"Not enough history: T={T}, need > {lookback+1}.")

    Sigma_series = np.zeros((T, n, n))
    for t in range(T):
        start = max(0, t - lookback)
        window = R[start:t+1, :]

        if window.shape[0] > 2:
            Sigma = np.cov(window, rowvar=False)
        else:
            Sigma = np.eye(n) * 1e-6

        Sigma_series[t] = Sigma * annualize

    return Sigma_series


# ============================================================
# 4) Alpha models: simple vs AR(1)
# ============================================================
def mu_series_simple(R, lookback_mu=60, mode="momentum", annualize=252):
    """
    Simple predictor:
      momentum: mean of last K returns
      mean_reversion: negative mean of last K returns
    Uses only data up to time t (inclusive).
    """
    T, n = R.shape
    mu = np.zeros((T, n))
    for t in range(T):
        start = max(0, t - lookback_mu + 1)
        window = R[start:t+1, :]
        m = np.nanmean(window, axis=0)
        if mode == "mean_reversion":
            m = -m
        mu[t] = m * annualize
    return mu


def mu_series_ar1(R, lookback_model=252, annualize=252, smooth_alpha=0.20):
    """
    Rolling AR(1) per asset via OLS:
        r_tau = a + b * r_{tau-1} + eps
    Forecast at time t:
        mu_t = a_hat + b_hat * r_t

    Optional smoothing:
        mu_sm[t] = alpha * mu[t] + (1-alpha) * mu_sm[t-1]
    """
    T, n = R.shape
    mu_raw = np.zeros((T, n))

    for t in range(T):
        start = max(1, t - lookback_model + 1)
        window = R[start:t+1, :]
        if window.shape[0] < 3:
            mu_raw[t] = 0.0
            continue

        X = window[:-1, :]
        Y = window[1:, :]

        for i in range(n):
            x = X[:, i]
            y = Y[:, i]
            A = np.column_stack([np.ones_like(x), x])
            theta, *_ = np.linalg.lstsq(A, y, rcond=None)
            a, b = theta
            mu_raw[t, i] = (a + b * R[t, i]) * annualize

    if smooth_alpha is None or smooth_alpha <= 0.0:
        return mu_raw

    mu_sm = np.zeros_like(mu_raw)
    mu_sm[0] = mu_raw[0]
    for t in range(1, T):
        mu_sm[t] = smooth_alpha * mu_raw[t] + (1.0 - smooth_alpha) * mu_sm[t - 1]
    return mu_sm


# ============================================================
# 5) PSD enforcement
# ============================================================
def make_psd(Sigma, ridge=1e-3, eps=1e-8):
    S = 0.5 * (Sigma + Sigma.T)
    S = S + ridge * np.eye(S.shape[0]) + eps * np.eye(S.shape[0])
    return cp.psd_wrap(cp.Constant(S))


# ============================================================
# 6) Constraints
# ============================================================
def add_constraints(w, long_only=True, w_min=None, w_max=None, fully_invested=True):
    cons = []
    n = w.shape[0]
    if fully_invested:
        cons.append(cp.sum(w) == 1)
    if long_only:
        cons.append(w >= 0)
    if w_min is not None:
        cons.append(w >= w_min * np.ones(n))
    if w_max is not None:
        cons.append(w <= w_max * np.ones(n))
    return cons


# ============================================================
# 7) Baseline one-step optimizer
# ============================================================
def solve_one_step(mu, Sigma, w_prev, gamma=5.0, lam=0.06, w_max=0.10, ridge=1e-3):
    n = len(mu)
    w = cp.Variable(n)

    turnover = cp.norm1(w - w_prev)
    Sigma_psd = make_psd(Sigma, ridge=ridge)

    obj = cp.Maximize(
        mu @ w
        - gamma * cp.quad_form(w, Sigma_psd)
        - lam * turnover
    )

    prob = cp.Problem(obj, add_constraints(w, long_only=True, w_max=w_max))
    prob.solve(solver=cp.SCS, warm_start=True, verbose=False)

    if w.value is None:
        return w_prev.copy()

    return np.asarray(w.value).reshape(-1)


def run_one_step(mu_series, Sigma_series, gamma=5.0, lam=0.06, w_max=0.10, ridge=1e-3):
    T, n = mu_series.shape
    w_prev = np.ones(n) / n
    W = np.zeros((T, n))

    for t in range(T):
        w_t = solve_one_step(
            mu_series[t], Sigma_series[t], w_prev,
            gamma=gamma, lam=lam, w_max=w_max, ridge=ridge
        )
        W[t] = w_t
        w_prev = w_t

    return W


# ============================================================
# 8) Advanced convex optimizer
# ============================================================
def solve_advanced_one_step(
    mu,
    Sigma,
    w_prev,
    w_bench,
    F=None,
    gamma=5.0,
    eta=2.0,
    lam=0.05,
    kappa=0.10,
    w_max=0.10,
    ridge=1e-3
):
    n = len(mu)
    w = cp.Variable(n)

    Sigma_psd = make_psd(Sigma, ridge=ridge)

    turnover = cp.norm1(w - w_prev)
    abs_risk = cp.quad_form(w, Sigma_psd)
    te_risk = cp.quad_form(w - w_bench, Sigma_psd)

    vol = np.sqrt(np.maximum(np.diag(Sigma), 1e-10))
    D = np.diag(vol)
    robust_penalty = cp.norm2(D @ w)

    obj = cp.Maximize(
        mu @ w
        - gamma * abs_risk
        - eta * te_risk
        - lam * turnover
        - kappa * robust_penalty
    )

    cons = [
        cp.sum(w) == 1,
        w >= 0,
        w <= w_max
    ]

    if F is not None:
        cons.append(F.T @ (w - w_bench) == 0)

    prob = cp.Problem(obj, cons)

    try:
        prob.solve(solver=cp.SCS, warm_start=True, verbose=False)
    except Exception:
        return w_prev.copy()

    if w.value is None:
        return w_prev.copy()

    w_out = np.asarray(w.value).reshape(-1)

    # optional cleanup for tiny numerical negatives / sum drift
    w_out = np.maximum(w_out, 0)
    s = w_out.sum()
    if s > 0:
        w_out = w_out / s

    return w_out


def run_advanced_one_step(
    mu_series,
    Sigma_series,
    gamma=5.0,
    eta=2.0,
    lam=0.05,
    kappa=0.10,
    w_max=0.10,
    ridge=1e-3,
    F=None
):
    T, n = mu_series.shape
    w_prev = np.ones(n) / n
    w_bench = np.ones(n) / n   # equal-weight benchmark
    W = np.zeros((T, n))

    for t in range(T):
        w_t = solve_advanced_one_step(
            mu=mu_series[t],
            Sigma=Sigma_series[t],
            w_prev=w_prev,
            w_bench=w_bench,
            F=F,
            gamma=gamma,
            eta=eta,
            lam=lam,
            kappa=kappa,
            w_max=w_max,
            ridge=ridge
        )
        W[t] = w_t
        w_prev = w_t

    return W


# ============================================================
# 9) Helpers: turnover, equity, stats
# ============================================================
def turnover_from_weights(W):
    return np.sum(np.abs(W[1:] - W[:-1]), axis=1)


def equity_from_weights_and_returns(R_daily, W, cost_rate=0.0005):
    """
    R_daily[t] is return from t-1 -> t.
    Use W[t-1] for R_daily[t] (causal).
    Transaction costs charged on turnover from W[t-1] -> W[t].
    """
    rp_gross = np.sum(R_daily[1:] * W[:-1], axis=1)
    to = turnover_from_weights(W)
    rp_net = rp_gross - cost_rate * to
    eq_gross = np.cumprod(1.0 + rp_gross)
    eq_net = np.cumprod(1.0 + rp_net)
    return eq_gross, eq_net, rp_gross, rp_net, to


def max_drawdown(equity_curve):
    peak = np.maximum.accumulate(equity_curve)
    dd = equity_curve / peak - 1.0
    return np.min(dd)


def annualized_sharpe(rp, periods_per_year=252):
    rp = np.asarray(rp)
    vol = np.std(rp)
    if vol < 1e-12:
        return np.nan
    return np.sqrt(periods_per_year) * np.mean(rp) / vol


def summarize_strategy(name, W, R_daily, cost_rate):
    eq_g, eq_n, rp_g, rp_n, to = equity_from_weights_and_returns(R_daily, W, cost_rate=cost_rate)
    return {
        "name": name,
        "final_equity_gross": eq_g[-1],
        "final_equity_net": eq_n[-1],
        "sharpe_gross": annualized_sharpe(rp_g),
        "sharpe_net": annualized_sharpe(rp_n),
        "max_dd_gross": max_drawdown(eq_g),
        "max_dd_net": max_drawdown(eq_n),
        "avg_turnover": np.mean(to),
        "avg_max_weight": np.mean(np.max(W, axis=1)),
        "eq_gross": eq_g,
        "eq_net": eq_n,
        "turnover": to,
    }


# ============================================================
# 10) Run comparison
# ============================================================
if __name__ == "__main__":
    # ---------------- settings ----------------
    seed = 7
    n_assets = 50

    lookback_sigma = 252
    K_simple = 60
    ar_window = 252
    annualize = 252

    # baseline optimizer params
    gamma_base = 5.0
    lam_simple_base = 0.06
    lam_ar1_base = 0.20

    # advanced optimizer params
    gamma_adv = 5.0
    eta_adv = 2.0
    lam_simple_adv = 0.06
    lam_ar1_adv = 0.20
    kappa_adv = 0.10

    w_max = 0.10
    ridge = 1e-3

    cost_rate = 0.0005
    smooth_alpha = 0.20

    # ---------------- load data ----------------
    data, chosen_assets = load_quantiacs_universe(seed=seed, n_assets=n_assets)
    print("Chosen assets (first 10):", chosen_assets[:10], "... total:", len(chosen_assets))

    times, R_daily = compute_returns_matrix(data, chosen_assets, use_log_returns=False)

    # ---------------- risk model ----------------
    Sigma_series = compute_sigma_series(R_daily, lookback=lookback_sigma, annualize=annualize)

    # ---------------- alpha models ----------------
    mu_simple = mu_series_simple(
        R_daily,
        lookback_mu=K_simple,
        mode="momentum",
        annualize=annualize
    )

    mu_ar1 = mu_series_ar1(
        R_daily,
        lookback_model=ar_window,
        annualize=annualize,
        smooth_alpha=smooth_alpha
    )

    # ---------------- baseline optimizer ----------------
    W_simple_base = run_one_step(
        mu_simple, Sigma_series,
        gamma=gamma_base, lam=lam_simple_base, w_max=w_max, ridge=ridge
    )

    W_ar1_base = run_one_step(
        mu_ar1, Sigma_series,
        gamma=gamma_base, lam=lam_ar1_base, w_max=w_max, ridge=ridge
    )

    # ---------------- advanced optimizer ----------------
    W_simple_adv = run_advanced_one_step(
        mu_simple, Sigma_series,
        gamma=gamma_adv, eta=eta_adv, lam=lam_simple_adv, kappa=kappa_adv,
        w_max=w_max, ridge=ridge, F=None
    )

    W_ar1_adv = run_advanced_one_step(
        mu_ar1, Sigma_series,
        gamma=gamma_adv, eta=eta_adv, lam=lam_ar1_adv, kappa=kappa_adv,
        w_max=w_max, ridge=ridge, F=None
    )

    # ---------------- sanity checks ----------------
    print("\n=== Sanity checks ===")
    print("Any NaNs in W_simple_base?", np.isnan(W_simple_base).any())
    print("Any NaNs in W_ar1_base?   ", np.isnan(W_ar1_base).any())
    print("Any NaNs in W_simple_adv? ", np.isnan(W_simple_adv).any())
    print("Any NaNs in W_ar1_adv?    ", np.isnan(W_ar1_adv).any())

    print("\nAvg max weight:")
    print("Simple baseline:", np.mean(W_simple_base.max(axis=1)))
    print("AR1 baseline:   ", np.mean(W_ar1_base.max(axis=1)))
    print("Simple advanced:", np.mean(W_simple_adv.max(axis=1)))
    print("AR1 advanced:   ", np.mean(W_ar1_adv.max(axis=1)))

    print("\nAvg turnover:")
    print("Simple baseline:", np.mean(turnover_from_weights(W_simple_base)))
    print("AR1 baseline:   ", np.mean(turnover_from_weights(W_ar1_base)))
    print("Simple advanced:", np.mean(turnover_from_weights(W_simple_adv)))
    print("AR1 advanced:   ", np.mean(turnover_from_weights(W_ar1_adv)))

    # ---------------- summarize ----------------
    results = []
    results.append(summarize_strategy("Simple + Baseline", W_simple_base, R_daily, cost_rate))
    results.append(summarize_strategy("AR1 + Baseline", W_ar1_base, R_daily, cost_rate))
    results.append(summarize_strategy("Simple + Advanced", W_simple_adv, R_daily, cost_rate))
    results.append(summarize_strategy("AR1 + Advanced", W_ar1_adv, R_daily, cost_rate))

    print("\n=== Summary table ===")
    print(f"{'Strategy':<20} {'Final Net':>12} {'Sharpe Net':>12} {'MaxDD Net':>12} {'Avg TO':>12}")
    for r in results:
        print(f"{r['name']:<20} {r['final_equity_net']:>12.4f} {r['sharpe_net']:>12.4f} {r['max_dd_net']:>12.4f} {r['avg_turnover']:>12.4f}")

    # ============================================================
    # 11) Plots
    # ============================================================

    # Gross equity
    plt.figure(figsize=(10, 5))
    for r in results:
        plt.plot(r["eq_gross"], label=r["name"])
    plt.title("Gross equity comparison")
    plt.xlabel("t")
    plt.ylabel("equity")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Net equity
    plt.figure(figsize=(10, 5))
    for r in results:
        plt.plot(r["eq_net"], label=r["name"])
    plt.title(f"Net equity comparison (cost_rate={cost_rate:.4f})")
    plt.xlabel("t")
    plt.ylabel("equity")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Turnover
    plt.figure(figsize=(10, 5))
    for r in results:
        plt.plot(r["turnover"], label=r["name"])
    plt.title("Turnover comparison: sum |Δw|")
    plt.xlabel("t")
    plt.ylabel("turnover")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Compare only alpha models within baseline
    plt.figure(figsize=(10, 5))
    plt.plot(results[0]["eq_net"], label="Simple + Baseline")
    plt.plot(results[1]["eq_net"], label="AR1 + Baseline")
    plt.title("Original notebook comparison under baseline optimizer")
    plt.xlabel("t")
    plt.ylabel("net equity")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Compare only alpha models within advanced optimizer
    plt.figure(figsize=(10, 5))
    plt.plot(results[2]["eq_net"], label="Simple + Advanced")
    plt.plot(results[3]["eq_net"], label="AR1 + Advanced")
    plt.title("Simple vs AR(1) under advanced convex optimizer")
    plt.xlabel("t")
    plt.ylabel("net equity")
    plt.legend()
    plt.tight_layout()
    plt.show()